In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import skimage

In [ ]:
# sample image
img = skimage.data.coins()
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
#  use gradient magnitude as edge weights
smoothed = skimage.filters.gaussian(img, sigma=1)
edges = skimage.filters.sobel(smoothed)
edges = skimage.filters.gaussian(edges, sigma=4)
edges = edges / np.max(edges)
height_map = edges


# plot height map
plt.imshow(height_map, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# create graph with 8-connected pixels
def eight_connected_grid_graph(shape, edge_weight_fn):
    
    def yield_edges():
        for x in range(shape[0]):
            for y in range(shape[1]):
                for dx in (-1, 0, 1):
                    for dy in (-1, 0, 1):
                        if dx == 0 and dy == 0:
                            continue
                        nx_, ny_ = x + dx, y + dy
                        if 0 <= nx_ < shape[0] and 0 <= ny_ < shape[1]:
                            if nx_ > x or (nx_ == x and ny_ > y):
                                yield (x, y), (nx_, ny_), edge_weight_fn(x, y, nx_, ny_)
    g = nx.Graph()
    for u, v, w in yield_edges():
        g.add_edge(u, v, weight=w)
    return g



def edge_weight_fn(x1, y1, x2, y2):
    return (height_map[x1, y1] + height_map[x2, y2]) + 0.1

g = eight_connected_grid_graph(height_map.shape[0:2], edge_weight_fn)

In [ ]:
# interactive ipycanvas widget to allow for clicking on the graph
from ipycanvas import Canvas

canvas = Canvas(width=height_map.shape[1], height=height_map.shape[0])
points = []

def draw_point(x, y, color='red', r=3):
    canvas.fill_style = color
    canvas.fill_circle(x, y, r)

def on_click(x, y):
    global points
    node = (int(y), int(x))
    if len(points) == 2:
        points = []
        canvas.put_image_data(height_map*255, 0, 0)
    points.append(node)

    draw_point(x, y, 'red' if len(points) == 1 else 'blue')

    if len(points) == 2:
        path = nx.shortest_path(g, source=points[0], target=points[1], weight='weight')
        print(f'Shortest path length: {len(path)}')

        for i in range(len(path) - 1):
            y1, x1 = path[i]
            y2, x2 = path[i + 1]
            canvas.stroke_style = 'green'
            canvas.line_width = 2
            canvas.stroke_line(x1, y1, x2, y2)

canvas.on_mouse_down(on_click)
canvas.put_image_data(height_map*255, 0, 0)
display(canvas)